In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

/Users/martindufour/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Function 6
print('Function 6')
func6_inputs = np.load('./initial_data/function_6/initial_inputs.npy')
print(func6_inputs)

func6_outputs = np.load('./initial_data/function_6/initial_outputs.npy')
print(func6_outputs)
print('/n')

Function 6
[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0

In [3]:
# Week 1 Input and Output Data
week_1_inputs = [ np.array([0.155793, 0.528435]), np.array([0.224164, 0.812385]), np.array([0.830919, 0.158523, 0.528406]), np.array([0.912533, 0.052672, 0.771239, 0.219812]), np.array([0.234189, 0.83648 , 0.884484, 0.873516]), np.array([0.490808, 0.618683, 0.277824, 0.900494, 0.106596]), np.array([0.067896, 0.486672, 0.255422, 0.215118, 0.427428, 0.72097 ]), np.array([0.061447, 0.062956, 0.029929, 0.036786, 0.407935, 0.795055, 0.496307,0.888085]) ]
week_1_outputs = [np.float64(1.5311489892413605e-58), np.float64(0.04614596805685454), np.float64(-0.04591165123945737), np.float64(-24.387232512869755), np.float64(1049.4420694211206), np.float64(-0.849252655626155), np.float64(1.3793294734939503), np.float64(9.598780741169)]

week_2_inputs = [np.array([0.946399, 0.18731 ]), np.array([0.712637, 0.921564]), np.array([0.765104, 0.052672, 0.438597]), np.array([0.380965, 0.771701, 0.089479, 0.536968]), np.array([0.219189, 0.85148 , 0.874484, 0.883516]), np.array([0.114755, 0.697421, 0.354179, 0.887624, 0.589139]), np.array([0.070896, 0.484672, 0.259422, 0.216118, 0.427428, 0.72297 ]), np.array([0.062447, 0.061956, 0.030929, 0.035786, 0.408935, 0.794055, 0.497307, 0.887085])]
week_2_outputs = [np.float64(2.338449488843798e-206), np.float64(0.5728372778652475), np.float64(-0.10235480701941752), np.float64(-13.368584815829887), np.float64(1109.9883069580462), np.float64(-1.492909868264592), np.float64(1.3944037721068683), np.float64(9.598978827169)]

week_3_inputs = [np.array([0.583738, 0.706798]), np.array([0.699637, 0.928564]), np.array([0.123457, 0.876543, 0.5     ]), np.array([0.230785, 0.914568, 0.102938, 0.657322]), np.array([0.214189, 0.85648 , 0.869484, 0.888516]), np.array([0.051235, 0.987654, 0.43211 , 0.123457, 0.765432]), np.array([0.071896, 0.483672, 0.261422, 0.217118, 0.426428, 0.72497 ]), np.array([0.063447, 0.060956, 0.031929, 0.034786, 0.409935, 0.793055, 0.498307, 0.886085])]
week_3_outputs = [np.float64(9.817718646044271e-07), np.float64(0.49978778689465564), np.float64(-0.04054152149606349), np.float64(-21.112987453792396), np.float64(1132.5255136709882), np.float64(-2.4679805862566795), np.float64(1.4059619293543582), np.float64(9.599155713169)]

week_4_inputs = [np.array([0.867072, 0.913241]), np.array([0.83604 , 0.696071]), np.array([0.447658, 0.395195, 0.505344]), np.array([0.422706, 0.385497, 0.37529 , 0.410833]), np.array([0.157706, 0.912326, 0.830158, 0.925715]), np.array([0.275401, 0.      , 0.568079, 1.      , 0.121326]), np.array([0.124679, 0.389027, 0.389012, 0.23686 , 0.379036, 0.802247]), np.array([0.077447, 0.21029 , 0.114032, 0.159535, 0.695924, 0.531316,0.178973, 0.57168 ])]
week_4_outputs = [np.float64(-1.0508862613282513e-96), np.float64(0.20657541488475104), np.float64(-0.03229682155733878), np.float64(0.4964978124935766), np.float64(1445.380906735266), np.float64(-0.8152779914672599), np.float64(2.0217812896063525), np.float64(9.987130922543)]

week_5_inputs = [np.array([0.065052, 0.948886]), np.array([0.388677, 0.271349]), np.array([1.      , 0.136717, 0.850593]), np.array([0.908266, 0.239562, 0.144895, 0.489453]), np.array([0.115614, 0.955641, 0.820859, 0.938174]), np.array([0.568442, 0.      , 1.      , 1.      , 1.      ]), np.array([0.134015, 0.028783, 0.755137, 0.62031 , 0.70408 , 0.212964]), np.array([0.127008, 0.292876, 0.06967 , 0.277582, 0.553407, 0.547258,0.220835, 0.443146])]
week_5_outputs = [np.float64(2.0262778967114778e-283), np.float64(0.016418658339648333), np.float64(-0.054388754089278846), np.float64(-17.161465002411145), np.float64(1779.8600577462366), np.float64(-1.9906490554141107), np.float64(0.052467603616080494), np.float64(9.9149654065019)]

week_6_inputs = [np.array([0.17701 , 0.088703]), np.array([0.593592, 0.679102]), np.array([0.7536, 1.    , 1.    ]), np.array([0.370348, 0.379959, 0.430216, 0.444351]), np.array([0.169493, 0.556801, 0.936155, 0.69603 ]), np.array([0.366464, 0.316099, 1.      , 1.      , 0.      ]), np.array([0.123574, 0.270452, 0.482301, 0.208163, 0.330398, 0.872733]), np.array([0.3191  , 0.828915, 0.037008, 0.59627 , 0.230009, 0.120567,0.076953, 0.696289])]
week_6_outputs = [np.float64(-2.3725238219366144e-119), np.float64(0.06836721478932847), np.float64(-0.48310415434111403), np.float64(0.18076540708623456), np.float64(282.83820524691396), np.float64(-0.8214281898088153), np.float64(2.075605759888563), np.float64(8.8803427965834)]

week_7_inputs = [np.array([0.065052, 0.948886]), np.array([0.140924, 0.802197]), np.array([1., 1., 0.]), np.array([0.32078 , 0.186519, 0.040775, 0.590893]), np.array([0.548734, 0.691895, 0.651961, 0.224269]), np.array([0.368433, 0.      , 1.      , 1.      , 0.428022]), np.array([0.139689, 0.317433, 0.463608, 0.250431, 0.325485, 0.810756]), np.array([0.      , 0.202823, 0.231703, 0.      , 1.      , 1.      ,
       0.288474, 1.      ])]
week_7_outputs = [np.float64(2.0262778967114778e-283), np.float64(-0.10826299524356352), np.float64(-0.16354530625442043), np.float64(-11.58523458824008), np.float64(1.9931553503870212), np.float64(-1.2796687884296385), np.float64(2.462201676843452), np.float64(9.622023932692)]

week_8_inputs = [np.array([0.000788, 0.033717]), np.array([0.914607, 0.789979]), np.array([0.317253, 0.002183, 0.963506]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.078217, 0.973099, 0.868006, 0.933352]), np.array([0.426158, 0.348959, 0.616644, 0.692851, 0.024814]), np.array([0.269889, 0.346609, 0.467584, 0.256632, 0.302654, 0.800092]), np.array([0.066269, 0.029193, 0.143059, 0.209133, 0.840619, 0.604919,
       0.230297, 0.701468])]
week_8_outputs = [np.float64(1.5608341712501477e-228), np.float64(0.0347797753016137), np.float64(-0.3756702789549372), np.float64(-33.661790988299735), np.float64(2151.3700669834334), np.float64(-0.23000336822278494), np.float64(2.4516632535923746), np.float64(9.9644234438351)]

week_9_inputs = [np.array([0.000924, 0.003116]), np.array([0.914607, 0.789979]), np.array([0.047574, 0.998325, 0.999125]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.080698, 0.971918, 0.842327, 0.954471]), np.array([0.942348, 0.037756, 0.105925, 0.010348, 0.997595]), np.array([0.090198, 0.680559, 0.851748, 0.080076, 0.298474, 0.701193]), np.array([0.95983 , 0.002073, 0.011779, 0.227207, 0.571611, 0.031204,
       0.24468 , 0.055697])]
week_9_outputs = [np.float64(7.25285761175276e-246), np.float64(0.05670257386671321), np.float64(-0.48158498276260003), np.float64(-33.661790988299735), np.float64(2156.522419821577), np.float64(-3.026993482898997), np.float64(0.6117443854593981), np.float64(8.1721431718416)]

week_10_inputs = [np.array([0.17701 , 0.088703]), np.array([0.593592, 0.679102]), np.array([0.7536, 1.    , 1.    ]), np.array([0.370348, 0.379959, 0.430216, 0.444351]), np.array([0.169493, 0.556801, 0.936155, 0.69603 ]), np.array([0.366464, 0.316099, 1.      , 1.      , 0.      ]), np.array([0.123574, 0.270452, 0.482301, 0.208163, 0.330398, 0.872733]), np.array([0.3191  , 0.828915, 0.037008, 0.59627 , 0.230009, 0.120567,
       0.076953, 0.696289])] 
week_10_outputs = [np.float64(0.0), np.float64(0.1625184332362701), np.float64(-0.13457392031599885), np.float64(-30.133085064011215), np.float64(1769.6901405375768), np.float64(-2.7724961974553874), np.float64(1.9208154093840464), np.float64(9.9296060847489)]

week_11_inputs = [np.array([0.37454 , 0.950714]), np.array([0.41611 , 0.666998]), np.array([0.997078, 0.475121, 0.651523]), np.array([0.00109 , 0.902069, 0.972212, 0.16754 ]), np.array([0.019324, 0.941663, 0.833048, 0.950636]), np.array([0.973398, 0.011482, 0.130865, 0.931063, 0.997256]), np.array([0.215391, 0.288991, 0.602008, 0.322698, 0.265768, 0.811541]), np.array([0.16109 , 0.049964, 0.210829, 0.130137, 0.965223, 0.408414,
       0.113403, 0.289162])]
week_11_outputs = [np.float64(-1.560646704467778e-117), np.float64(-0.1334547156009971), np.float64(-0.10208280924057045), np.float64(-31.74483921038956), np.float64(1827.9676185989063), np.float64(-2.432448557520746), np.float64(2.4819294367786457), np.float64(9.9158368797091)]

week_12_inputs = [np.array([0.001   , 0.999576]), np.array([0.45253, 0.6577 ]), np.array([0.998539, 0.483668, 0.657653]), np.array([0.018818, 0.998904, 0.999658, 0.998016]), np.array([0.064807, 0.978215, 0.856404, 0.971059]), np.array([0.898391, 0.002198, 0.019804, 0.999554, 0.917075]), np.array([0.211376, 0.335246, 0.510659, 0.146838, 0.289515, 0.653795]), np.array([4.18000e-04, 1.05120e-02, 1.95045e-01, 2.11750e-01, 7.88533e-01,
       9.00374e-01, 2.41123e-01, 3.34541e-01])]
week_12_outputs = [np.float64(0.0), np.float64(0.3536071649472596), np.float64(-0.09423686717649356), np.float64(-47.62748092409266), np.float64(2449.5648412319215), np.float64(-2.500439911687528), np.float64(2.549261085159255), np.float64(9.7734101935864)]

In [4]:
# Function 6
print('Function 6')
# Load inputs from previous run
# Loads initial data
week_0_func6_inputs = np.load('./initial_data/function_6/initial_inputs.npy')
week_0_func6_outputs = np.load('./initial_data/function_6/initial_outputs.npy')

print(f'Shape of initial input data: {week_0_func6_inputs.shape}')
print(f'Shape of initial output data: {week_0_func6_outputs.shape}')

print(f'Week 1 inputs: {week_1_inputs[5]}')
print(f'Week 2 inputs: {week_2_inputs[5]}')
print(f'Week 3 inputs: {week_3_inputs[5]}')
print(f'Week 4 inputs: {week_4_inputs[5]}')
print(f'Week 5 inputs: {week_5_inputs[5]}')
print(f'Week 6 inputs: {week_6_inputs[5]}')
print(f'Week 7 inputs: {week_7_inputs[5]}')
print(f'Week 8 inputs: {week_8_inputs[5]}')
print(f'Week 9 inputs: {week_9_inputs[5]}')
print(f'Week 10 inputs: {week_10_inputs[5]}')
print(f'Week 11 inputs: {week_11_inputs[5]}')
print(f'Week 12 inputs: {week_12_inputs[5]}')

combined_func6_inputs = np.vstack([
    week_0_func6_inputs,
    week_1_inputs[5],
    week_2_inputs[5],
    week_3_inputs[5],
    week_4_inputs[5],
    week_5_inputs[5],
    week_6_inputs[5],
    week_7_inputs[5],
    week_8_inputs[5],
    week_9_inputs[5],
    week_10_inputs[5],
    week_11_inputs[5],
    week_12_inputs[5]
])
print(f'Number of input data points: {len(combined_func6_inputs)}')
print('Combined input data')
print(combined_func6_inputs)

# Load outputs from previous run
week_func6_output = week_1_outputs[5]
combined_func6_outputs = np.concatenate([
    week_0_func6_outputs,
    [week_1_outputs[5]],
    [week_2_outputs[5]],
    [week_3_outputs[5]],
    [week_4_outputs[5]],
    [week_5_outputs[5]],
    [week_6_outputs[5]],
    [week_7_outputs[5]],
    [week_8_outputs[5]],
    [week_9_outputs[5]],
    [week_10_outputs[5]],
    [week_11_outputs[5]],
    [week_12_outputs[5]]
])
print(f'Number of output data points: {len(combined_func6_outputs)}')
print('Combined output data')
print(combined_func6_outputs)

Function 6
Shape of initial input data: (20, 5)
Shape of initial output data: (20,)
Week 1 inputs: [0.490808 0.618683 0.277824 0.900494 0.106596]
Week 2 inputs: [0.114755 0.697421 0.354179 0.887624 0.589139]
Week 3 inputs: [0.051235 0.987654 0.43211  0.123457 0.765432]
Week 4 inputs: [0.275401 0.       0.568079 1.       0.121326]
Week 5 inputs: [0.568442 0.       1.       1.       1.      ]
Week 6 inputs: [0.366464 0.316099 1.       1.       0.      ]
Week 7 inputs: [0.368433 0.       1.       1.       0.428022]
Week 8 inputs: [0.426158 0.348959 0.616644 0.692851 0.024814]
Week 9 inputs: [0.942348 0.037756 0.105925 0.010348 0.997595]
Week 10 inputs: [0.366464 0.316099 1.       1.       0.      ]
Week 11 inputs: [0.973398 0.011482 0.130865 0.931063 0.997256]
Week 12 inputs: [0.898391 0.002198 0.019804 0.999554 0.917075]
Number of input data points: 32
Combined input data
[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 

In [5]:
### ====== OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 6 ======
# Import the OptunaBayesianOptimizer class
import sys
sys.path.insert(0, './bayesian_optimization_challenge-md')
from bo_optuna import OptunaBayesianOptimizer

# Create optimizer instance with initial Function 6 data
print("=" * 60)
print("OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 6")
print("=" * 60)

X_func6_initial = week_0_func6_inputs
y_func6_initial = week_0_func6_outputs
bounds_func6 = [(0, 1)] * X_func6_initial.shape[1]

optimizer_func6 = OptunaBayesianOptimizer(
    X_initial=X_func6_initial,
    y_initial=y_func6_initial,
    bounds=bounds_func6,
    optimize_hp=True,  # Enable hyperparameter tuning
    random_state=42,
    acquisition="ucb"
)

print(f"\nInitial training data shape: X={optimizer_func6.X_train.shape}, y={optimizer_func6.y_train.shape}")
print(f"Initial best observation: {optimizer_func6.get_best_observation()[1]:.6e}")


OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 6

Initial training data shape: X=(20, 5), y=(20,)
Initial best observation: -7.142649e-01


In [6]:
# Run Optuna-based BO for 8 weeks (8 iterations)
print("\nRunning Optuna-based BO for 8 iterations...")
print("-" * 60)

# Get all weekly data
weekly_data = [
    (week_1_inputs[5], week_1_outputs[5]),
    (week_2_inputs[5], week_2_outputs[5]),
    (week_3_inputs[5], week_3_outputs[5]),
    (week_4_inputs[5], week_4_outputs[5]),
    (week_5_inputs[5], week_5_outputs[5]),
    (week_6_inputs[5], week_6_outputs[5]),
    (week_7_inputs[5], week_7_outputs[5]),
    (week_8_inputs[5], week_8_outputs[5]),
    (week_9_inputs[5], week_9_outputs[5]),
    (week_10_inputs[5], week_10_outputs[5]),
    (week_11_inputs[5], week_11_outputs[5]),
    (week_12_inputs[5], week_12_outputs[5])
]

optuna_proposals_func6 = []
manual_best_func6 = week_0_func6_outputs.max()
optuna_best_func6 = y_func6_initial.max()

for week, (x_actual, y_actual) in enumerate(weekly_data, start=1):
    print(f"\nWeek {week}:")
    print(f"  Actual observation: y = {y_actual:.6e}")
    
    # Get Optuna proposal
    proposals = optimizer_func6.optimize(
        n_iterations=1,
        optimize_hp_every=1 if week % 2 == 0 else 0,  # Tune HP every other week
        optimize_hp_n_trials=30,
        acq_n_trials=100,
        verbose=True
    )
    
    x_proposed = proposals[0]
    optuna_proposals_func6.append(x_proposed)
    
    # Update optimizer with actual observation
    optimizer_func6.update(x_actual, y_actual)
    
    # Track best values
    manual_best_func6 = max(manual_best_func6, y_actual)
    optuna_best_func6 = max(optuna_best_func6, y_actual)
    
    print(f"  Best so far (Optuna): {optuna_best_func6:.6e}")

print("\n" + "=" * 60)
print("OPTUNA-BASED BO COMPLETED FOR FUNCTION 6")
print("=" * 60)


[I 2026-05-04 06:34:59,222] A new study created in memory with name: no-name-614cc6aa-50c3-4556-b1e0-bd7462a816e9
[I 2026-05-04 06:34:59,224] Trial 0 finished with value: 1.4085613938099784 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051, 'x3': 0.5986584841970366, 'x4': 0.15601864044243652}. Best is trial 0 with value: 1.4085613938099784.
[I 2026-05-04 06:34:59,225] Trial 1 finished with value: 5.118338165327688 and parameters: {'x0': 0.15599452033620265, 'x1': 0.05808361216819946, 'x2': 0.8661761457749352, 'x3': 0.6011150117432088, 'x4': 0.7080725777960455}. Best is trial 1 with value: 5.118338165327688.
[I 2026-05-04 06:34:59,228] Trial 2 finished with value: 2.9720415410612926 and parameters: {'x0': 0.020584494295802447, 'x1': 0.9699098521619943, 'x2': 0.8324426408004217, 'x3': 0.21233911067827616, 'x4': 0.18182496720710062}. Best is trial 1 with value: 5.118338165327688.
[I 2026-05-04 06:34:59,230] Trial 3 finished with value: 3.0772263


Running Optuna-based BO for 8 iterations...
------------------------------------------------------------

Week 1:
  Actual observation: y = -8.492527e-01


[I 2026-05-04 06:34:59,338] Trial 18 finished with value: 2.858788011781777 and parameters: {'x0': 0.2628961383260644, 'x1': 0.42229939592318305, 'x2': 0.41520661636330003, 'x3': 0.8696264277319194, 'x4': 0.32085017861981696}. Best is trial 8 with value: 6.874438881544713.
[I 2026-05-04 06:34:59,350] Trial 19 finished with value: 2.293133418279867 and parameters: {'x0': 0.5025776676557023, 'x1': 0.6439003482217635, 'x2': 0.0032878518931074192, 'x3': 0.010989189243771236, 'x4': 0.8145901286801429}. Best is trial 8 with value: 6.874438881544713.
[I 2026-05-04 06:34:59,360] Trial 20 finished with value: 4.698411361224123 and parameters: {'x0': 0.26690375207555295, 'x1': 0.4641664960449513, 'x2': 0.6092599562105125, 'x3': 0.7243280325525963, 'x4': 0.0047401196530895695}. Best is trial 8 with value: 6.874438881544713.
[I 2026-05-04 06:34:59,369] Trial 21 finished with value: 5.938881486275693 and parameters: {'x0': 0.13021190084848497, 'x1': 0.00035129100840898715, 'x2': 0.8360728147769022,

[Iteration 0] Proposed: [0.04190329 0.91016093 0.01521565 0.90371795 0.03332059], UCB: 11.931653
  Best so far (Optuna): -7.142649e-01

Week 2:
  Actual observation: y = -1.492910e+00


[I 2026-05-04 06:35:00,503] Trial 19 finished with value: 5.949337268796564 and parameters: {'x0': 0.44878231542062874, 'x1': 0.39385500788924593, 'x2': 0.9692698520487176, 'x3': 0.37397770574158445, 'x4': 0.880405186301052}. Best is trial 12 with value: 11.598173128765385.
[I 2026-05-04 06:35:00,514] Trial 20 finished with value: 7.885204073375435 and parameters: {'x0': 0.26690375207555295, 'x1': 0.09036331498967792, 'x2': 0.6092599562105125, 'x3': 0.6908570884162393, 'x4': 0.979558187877013}. Best is trial 12 with value: 11.598173128765385.
[I 2026-05-04 06:35:00,525] Trial 21 finished with value: 9.673015134666244 and parameters: {'x0': 0.3349040259984848, 'x1': 0.0008089091439480254, 'x2': 0.9430174461975479, 'x3': 0.840537191728312, 'x4': 0.7509972360125259}. Best is trial 12 with value: 11.598173128765385.
[I 2026-05-04 06:35:00,534] Trial 22 finished with value: 6.351711360505641 and parameters: {'x0': 0.5466838194683197, 'x1': 0.22398951188179878, 'x2': 0.9055659745469131, 'x3'

...

[Iteration 0] Proposed: [0.97339761 0.01148244 0.13086518 0.93106313 0.99725645], UCB: 8.558115
  Best so far (Optuna): -2.300034e-01

Week 11:
  Actual observation: y = -2.432449e+00


[I 2026-05-04 06:35:11,034] Trial 16 finished with value: 0.6471205338827781 and parameters: {'x0': 0.6090765478348508, 'x1': 0.852863139636479, 'x2': 0.3957489155016659, 'x3': 0.7960616323362711, 'x4': 0.5954555772697581}. Best is trial 6 with value: 6.782216775664484.
[I 2026-05-04 06:35:11,046] Trial 17 finished with value: 6.100933915053151 and parameters: {'x0': 0.8963645753766486, 'x1': 0.013140315717808437, 'x2': 0.1463562344868443, 'x3': 0.7200151578178959, 'x4': 0.873919462145916}. Best is trial 6 with value: 6.782216775664484.
[I 2026-05-04 06:35:11,054] Trial 18 finished with value: 2.5678907146839616 and parameters: {'x0': 0.7163913183156698, 'x1': 0.5311240303514603, 'x2': 0.6761082710217003, 'x3': 0.8940519332841532, 'x4': 0.6315745565372841}. Best is trial 6 with value: 6.782216775664484.
[I 2026-05-04 06:35:11,066] Trial 19 finished with value: 2.1869779990173734 and parameters: {'x0': 0.5437419248054699, 'x1': 0.40724981184005865, 'x2': 0.1834154272926376, 'x3': 0.7546

[Iteration 0] Proposed: [0.89839059 0.00219777 0.01980359 0.99955424 0.91707472], UCB: 10.160070
  Best so far (Optuna): -2.300034e-01

Week 12:
  Actual observation: y = -2.500440e+00


[I 2026-05-04 06:35:12,289] Trial 19 finished with value: 2.9064081905581087 and parameters: {'x0': 0.8573911300093487, 'x1': 0.5895985018457592, 'x2': 0.3279136801148529, 'x3': 0.12782432428264007, 'x4': 0.008052950768387133}. Best is trial 17 with value: 6.364285461465673.
[I 2026-05-04 06:35:12,300] Trial 20 finished with value: 4.842270327208361 and parameters: {'x0': 0.8964859698407046, 'x1': 0.8629273218532529, 'x2': 0.17153416809741334, 'x3': 0.29703160461961053, 'x4': 0.31987257359688775}. Best is trial 17 with value: 6.364285461465673.
[I 2026-05-04 06:35:12,310] Trial 21 finished with value: 4.888706076532627 and parameters: {'x0': 0.9345477013308665, 'x1': 0.8395392694827274, 'x2': 0.3416449239118234, 'x3': 0.3335836773923127, 'x4': 0.09248681917996644}. Best is trial 17 with value: 6.364285461465673.
[I 2026-05-04 06:35:12,321] Trial 22 finished with value: 4.1675465387248956 and parameters: {'x0': 0.9961746388383605, 'x1': 0.707194478566351, 'x2': 0.4115588080699067, 'x3':

[Iteration 0] Proposed: [0.03049268 0.07442473 0.0424105  0.08416805 0.00542894], UCB: 14.175046
  Best so far (Optuna): -2.300034e-01

OPTUNA-BASED BO COMPLETED FOR FUNCTION 6
